# 02 — Data Cleaning & Combined Dataframe

Merges all cleaned sources into one home-game dataframe ready for feature engineering.

| Source | Key columns |
|---|---|
| `gold_match.csv` | match info, result, target (`tickets_scanned`) |
| `gold_match_context.csv` | weather, calendar, promotions |
| `gold_match_tickets.csv` | ticket sales breakdown |
| `match_buzz.csv` | pre-match press article count |
| `match_trends.csv` | 7-day avg Google Trends interest |

In [7]:
import pandas as pd

## 1. Match data

In [8]:
raw_match = pd.read_csv('../data/raw/gold_match.csv', on_bad_lines='skip')

# Fix types
raw_match['match_date'] = pd.to_datetime(raw_match['match_date'])
raw_match['kickoff_time_local'] = pd.to_datetime(raw_match['kickoff_time_local'], format='%H:%M:%S').dt.time
raw_match['is_home_match'] = raw_match['is_home_match'].astype(bool)
raw_match['matchday'] = raw_match['matchday'].astype('Int64')

# Opponent from OHL's perspective (before dropping team columns)
raw_match['opponent'] = raw_match.apply(
    lambda r: r['away_team'] if r['is_home_match'] else r['home_team'], axis=1
)

# Result and goals from OHL's perspective
def ohl_result(row):
    if row['is_home_match']:
        return row['result_home']
    return {'W': 'L', 'L': 'W', 'D': 'D'}.get(row['result_home'])

raw_match['ohl_result']         = raw_match.apply(ohl_result, axis=1)
raw_match['ohl_goals_ft']       = raw_match.apply(lambda r: r['goals_home_ft'] if r['is_home_match'] else r['goals_away_ft'], axis=1)
raw_match['opponent_goals_ft']  = raw_match.apply(lambda r: r['goals_away_ft'] if r['is_home_match'] else r['goals_home_ft'], axis=1)

# Parse last H2H result
raw_match['last_h2h_result']         = raw_match['last_result_vs_opponent'].str.extract(r'^([WDL])')
raw_match['last_h2h_ohl_goals']      = raw_match['last_result_vs_opponent'].str.extract(r'(\d+)-\d+$').astype('Int64')
raw_match['last_h2h_opponent_goals'] = raw_match['last_result_vs_opponent'].str.extract(r'\d+-(\d+)$').astype('Int64')

# Keep only necessary columns
match_df = raw_match[[
    'match_id', 'match_date', 'kickoff_time_local', 'matchday', 'season',
    'stage', 'opponent', 'is_home_match', 'ohl_result',
    'ohl_goals_ft', 'opponent_goals_ft',
    'last_h2h_result', 'last_h2h_ohl_goals', 'last_h2h_opponent_goals',
    'tickets_scanned'
]].copy()

# Filter to home games only
match_df = match_df[match_df['is_home_match']].drop(columns='is_home_match').reset_index(drop=True)

print(f"Home matches: {len(match_df)}")
match_df.head()

Home matches: 71


,match_id,match_date,kickoff_time_local,matchday,season,stage,opponent,ohl_result,ohl_goals_ft,opponent_goals_ft,last_h2h_result,last_h2h_ohl_goals,last_h2h_opponent_goals,tickets_scanned
0,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2,2022/2023,Regular Season,Westerlo,W,2,0,NaN,<NA>,<NA>,5565.0
1,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,4,2022/2023,Regular Season,Club Brugge,L,0,3,L,1,4,7440.0
2,d65hmi7sq03yzr5he1k7ypus4,2022-08-27,18:15:00,6,2022/2023,Regular Season,KV Oostende,W,2,1,W,3,1,4489.0
3,d80mkemezkz16bqh6lbn8tlhw,2022-09-10,20:45:00,8,2022/2023,Regular Season,Sporting Charleroi,W,3,2,W,3,0,4508.0
4,dak40etbhbqsr1nxyt50qcg0k,2022-10-01,16:00:00,10,2022/2023,Regular Season,Union Saint-Gilloise,L,0,3,L,1,4,6290.0


## 2. Match context (weather, calendar, promotions)

In [9]:
raw_context = pd.read_csv('../data/raw/gold_match_context.csv', on_bad_lines='skip')

# Filter to home games
home_ids = raw_match[raw_match['is_home_match']]['match_id']
raw_context = raw_context[raw_context['match_id'].isin(home_ids)].reset_index(drop=True)

# Fix types
raw_context['match_date'] = pd.to_datetime(raw_context['match_date'])
bool_cols = ['is_weekend', 'is_midweek', 'is_public_holiday', 'is_school_holiday_flanders', 'has_promotion']
for col in bool_cols:
    raw_context[col] = raw_context[col].map({'true': True, 'false': False}).astype(bool)

# Binary flags
raw_context['has_campaign'] = raw_context['campaign_motto'].notna()
raw_context['has_snow']     = raw_context['weather_snowfall_cm'] > 0

context_df = raw_context[[
    'match_id',
    'weather_score', 'weather_temp_deviation', 'weather_rain_mm',
    'weather_windspeed_max_kmh', 'weather_sunshine_hours', 'has_snow',
    'is_weekend', 'is_midweek', 'academic_week',
    'is_public_holiday', 'is_school_holiday_flanders',
    'has_promotion', 'promo_tickets_total', 'has_campaign',
]].copy()

print(f"Context rows: {len(context_df)}")
context_df.head()

Context rows: 71


,match_id,weather_score,weather_temp_deviation,weather_rain_mm,weather_windspeed_max_kmh,weather_sunshine_hours,has_snow,is_weekend,is_midweek,academic_week,is_public_holiday,is_school_holiday_flanders,has_promotion,promo_tickets_total,has_campaign
0,d256yo3eng04m0fu7b4sl7wno,0.09,0.9,0.0,10.2,10.3,False,True,True,48,True,True,True,0,False
1,d4mn5ksbxuvnaww4pmommxhqs,0.59,5.9,0.0,16.9,11.1,False,True,True,50,True,True,True,0,False
2,d65hmi7sq03yzr5he1k7ypus4,-0.08,-0.8,0.0,13.7,3.9,False,True,True,52,True,True,True,0,False
3,d80mkemezkz16bqh6lbn8tlhw,-3.64,-0.6,8.6,23.5,4.1,False,True,True,2,True,True,True,0,False
4,dak40etbhbqsr1nxyt50qcg0k,-3.83,1.1,9.8,34.5,8.5,False,True,True,5,True,True,True,0,False


## 3. Ticket sales

In [10]:
tickets_df = pd.read_csv('../data/raw/gold_match_tickets.csv', on_bad_lines='skip')

# Ensure numeric cols are correct type
numeric_cols = tickets_df.columns.drop('match_id')
tickets_df[numeric_cols] = tickets_df[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

print(f"Ticket rows: {len(tickets_df)}")
tickets_df.head()

Ticket rows: 71


,match_id,tickets_sold_b2c,tickets_sold_b2b,tickets_sold_total,seasonpass_holders,tickets_trib1,tickets_trib2_thuis,tickets_trib2_uit,tickets_trib3,tickets_trib4
0,3kewmr690bhoid47mku9wfgno,1095,882,5988,4405,2088,427,1,1903,1569
1,3nrkjdhpc5zse1sdzt1onor2s,1807,953,6652,4405,2256,517,1,2307,1571
2,3rwouz4640n6ngfbwrwsqkp3o,2793,880,7385,4405,2405,572,17,2821,1570
3,3vk3btgbt7w7qy9hjp3f2934k,1981,1003,6929,4405,2441,584,0,2334,1570
4,3yp7glzujzj8nc07qp0sd0kr8,1936,925,6901,4405,2329,566,0,2437,1569


## 4. Pre-match buzz & Google Trends

In [11]:
buzz_df   = pd.read_csv('../data/cleaned/match_buzz.csv')
trends_df = pd.read_csv('../data/cleaned/match_trends.csv')

print(f"Buzz rows: {len(buzz_df)},  Trends rows: {len(trends_df)}")

Buzz rows: 132,  Trends rows: 142


## 5. Merge into combined dataframe

In [12]:
combined_df = (
    match_df
    .merge(context_df, on='match_id', how='left')
    .merge(tickets_df, on='match_id', how='left')
    .merge(buzz_df,    on='match_id', how='left')
    .merge(trends_df,  on='match_id', how='left')
)

combined_df = combined_df.sort_values('match_date').reset_index(drop=True)

print(f"Shape: {combined_df.shape}")
print(f"\nMissing values:")
print(combined_df.isnull().sum()[combined_df.isnull().sum() > 0])
combined_df.head()

Shape: (71, 39)

Missing values:
last_h2h_result            3
last_h2h_ohl_goals         3
last_h2h_opponent_goals    3
pre_match_article_count    4
dtype: int64


,match_id,match_date,kickoff_time_local,matchday,season,stage,opponent,ohl_result,ohl_goals_ft,opponent_goals_ft,...,tickets_sold_b2b,tickets_sold_total,seasonpass_holders,tickets_trib1,tickets_trib2_thuis,tickets_trib2_uit,tickets_trib3,tickets_trib4,pre_match_article_count,avg_ohl_interest_7d
0,d256yo3eng04m0fu7b4sl7wno,2022-07-30,18:15:00,2,2022/2023,Regular Season,Westerlo,W,2,0,...,764,5025,4032,1899,127,0,1427,1572,14.0,5.165714
1,d4mn5ksbxuvnaww4pmommxhqs,2022-08-14,18:30:00,4,2022/2023,Regular Season,Club Brugge,L,0,3,...,1284,6263,4032,2387,323,0,1982,1571,48.0,7.812857
2,d65hmi7sq03yzr5he1k7ypus4,2022-08-27,18:15:00,6,2022/2023,Regular Season,KV Oostende,W,2,1,...,823,5025,4032,1876,126,0,1453,1570,3.0,8.462857
3,d80mkemezkz16bqh6lbn8tlhw,2022-09-10,20:45:00,8,2022/2023,Regular Season,Sporting Charleroi,W,3,2,...,906,5438,4032,1897,560,0,1411,1570,8.0,7.291429
4,dak40etbhbqsr1nxyt50qcg0k,2022-10-01,16:00:00,10,2022/2023,Regular Season,Union Saint-Gilloise,L,0,3,...,970,5844,4032,2163,355,0,1757,1569,NaN,2.788571


In [13]:
combined_df.to_csv('../data/cleaned/combined_df.csv', index=False)
print('Saved.')

Saved.
